In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
from matplotlib import rcParams
import time
from datetime import datetime

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, BatchNorm
from torch_geometric.data import Data, DataLoader
from torch_geometric.utils import dense_to_sparse

# Sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.spatial.distance import pdist, squareform

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# HELPER FUNCTIONS

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    """Convert time series data to supervised learning format."""
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    
    # Input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    
    # Forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [ ]:
def create_feature_graph(data, correlation_threshold=0.3):
    """Create graph structure based on feature correlations."""
    # Calculate correlation matrix
    corr_matrix = np.corrcoef(data.T)
    
    # Create adjacency matrix based on correlation threshold
    adj_matrix = (np.abs(corr_matrix) > correlation_threshold).astype(float)
    np.fill_diagonal(adj_matrix, 1)  # Self-connections
    
    # Convert to edge list
    edge_index, edge_weights = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float))
    
    return edge_index, edge_weights, corr_matrix

In [ ]:
def create_graph_data_list(X, Y, window_size=48):
    """Convert time series data to graph format."""
    graph_data_list = []
    
    # Create base graph structure from feature correlations
    sample_data = X[0].reshape(-1, X.shape[-1])
    edge_index, edge_weights, _ = create_feature_graph(sample_data)
    
    for i in range(len(X)):
        # Node features: each feature becomes a node with temporal values
        node_features = X[i].T  # [features, timesteps]
        
        # Create graph data object
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=edge_index.long(),
            y=torch.tensor([Y[i]], dtype=torch.float32)
        )
        graph_data_list.append(data)
    
    return graph_data_list

# LOAD DATASET

In [ ]:
print("Loading dataset...")
dataset = pd.read_csv("eMalahleniIM.csv", sep=';', header=0, index_col=0)
values = dataset.values
print(f"Dataset shape: {dataset.shape}")
print(f"Columns: {list(dataset.columns)}")

# DATA PREPROCESSING

In [ ]:
print("Preprocessing data...")

# Ensure all data is float
values = values.astype('float32')

In [ ]:
# Normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

In [ ]:
# Create windowed data for temporal modeling
window_size = 48  # 24-hour window
X_windows = []
Y_windows = []

In [ ]:
for i in range(len(scaled) - window_size):
    X_windows.append(scaled[i:i+window_size])
    Y_windows.append(scaled[i+window_size, 0])  # Predict PM2.5 (column 0)

X = np.array(X_windows)  # [samples, timesteps, features]
Y = np.array(Y_windows)  # [samples]
n_features = scaled.shape[1]

print(f"Windowed X shape: {X.shape}, Y shape: {Y.shape}")

In [ ]:
# Convert to graph data
print("Creating graph data...")
graph_data = create_graph_data_list(X, Y, window_size)
print(f"Created {len(graph_data)} graph samples")

# TRAIN/VALIDATION/TEST SPLIT

In [ ]:
print("Splitting data...")
n_samples = len(graph_data)
indices = list(range(n_samples))

In [ ]:
# First split: separate test set
train_val_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

In [ ]:
# Second split: separate validation from training  
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.2, random_state=42)


In [ ]:
train_data = [graph_data[i] for i in train_idx]
val_data = [graph_data[i] for i in val_idx]
test_data = [graph_data[i] for i in test_idx]

print(f"Train: {len(train_data)} graphs")
print(f"Val: {len(val_data)} graphs")  
print(f"Test: {len(test_data)} graphs")

# GCN MODEL DEFINITION

In [ ]:
class GCNModel(nn.Module):
    """Graph Convolutional Network for air pollution prediction."""
    
    def __init__(self, n_features, hidden_dim=32, n_layers=4, dropout=0.1):
        super(GCNModel, self).__init__()
        
        self.n_layers = n_layers
        self.dropout = dropout
        
        # GCN layers
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # Input layer
        self.convs.append(GCNConv(n_features, hidden_dim))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Hidden layers
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Output layer
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Final prediction layers
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # GCN layers with residual connections
        for i in range(self.n_layers):
            x_residual = x if i > 0 and x.size(-1) == self.convs[i].out_channels else None
            
            x = self.convs[i](x, edge_index)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            
            # Residual connection
            if x_residual is not None:
                x = x + x_residual
        
        # Global pooling to get graph-level representation
        x = global_mean_pool(x, batch)
        
        # Final prediction
        out = self.predictor(x)
        
        return out.squeeze()

# MODEL SETUP

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Create data loaders
batch_size = 16
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# Initialize model
model = GCNModel(
    n_features=window_size,  # Temporal features per node
    hidden_dim=32,
    n_layers=4,
    dropout=0.1
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)


# TRAINING FUNCTIONS

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            out = model(batch)
            loss = criterion(out, batch.y)
            
            total_loss += loss.item()
            predictions.extend(out.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())
    
    return total_loss / len(data_loader), np.array(predictions), np.array(targets)

# TRAINING LOOP

In [ ]:
print("Starting GCN training...")
epochs = 150
patience = 15
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

start_time = time.time()

In [ ]:
for epoch in range(epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate
    val_loss, _, _ = evaluate_model(model, val_loader, criterion, device)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_gcn_model.pth')
    else:
        patience_counter += 1
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d}: Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')
    
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

training_time = time.time() - start_time
print(f'Training completed in {training_time:.2f} seconds')
print(f'Best validation loss: {best_val_loss:.6f}')

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_gcn_model.pth'))

# EVALUATION

In [ ]:
print("Evaluating on test set...")
test_loss, predictions, targets = evaluate_model(model, test_loader, criterion, device)

In [ ]:
# Calculate metrics
mse = mean_squared_error(targets, predictions)
mae = mean_absolute_error(targets, predictions) 
rmse = sqrt(mse)
r2 = r2_score(targets, predictions)

print(f"\nGCN Test Results:")
print(f"MSE:  {mse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.6f}")


# VISUALIZATIONS

In [ ]:
print("Creating visualizations...")

rcParams['font.weight'] = 'bold'
plt.figure(figsize=(15, 5))

# Training history
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(history['val_loss'], label='Val Loss', linewidth=2)
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('MSE Loss', fontweight='bold')
plt.title('GCN Training History', fontweight='bold', size=14)
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)

In [ ]:
# Predictions vs Actual
plt.subplot(1, 3, 2)
plt.scatter(targets, predictions, alpha=0.6, s=10, c='blue')
plt.plot([targets.min(), targets.max()], [targets.min(), targets.max()], 'r--', lw=2)
plt.xlabel('Actual PM2.5', fontweight='bold')
plt.ylabel('Predicted PM2.5', fontweight='bold')
plt.title('GCN: Predicted vs Actual', fontweight='bold', size=14)
plt.grid(True, alpha=0.3)

In [ ]:
# Time series plot
plt.subplot(1, 3, 3)
n_points = min(100, len(targets))
plt.plot(targets[:n_points], label='Actual', linewidth=2, alpha=0.8)
plt.plot(predictions[:n_points], label='Predicted', linewidth=1.5, alpha=0.8)
plt.xlabel('Time Steps', fontweight='bold')
plt.ylabel('Normalized PM2.5', fontweight='bold')
plt.title('GCN Time Series Prediction', fontweight='bold', size=14)
plt.legend()
plt.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('gcn_results.png', dpi=300, bbox_inches='tight')
plt.show()

# QUANTILE ANALYSIS

In [ ]:
print("Performing quantile analysis...")

# Calculate errors
errors = predictions.flatten() - targets

# Calculate quantiles based on actual values
quantiles, bins = pd.qcut(targets, q=10, duplicates='drop', retbins=True)

# Calculate average error for each quantile
quantile_errors = []
for i in range(len(bins) - 1):
    group_indices = np.where((targets >= bins[i]) & (targets < bins[i+1]))[0]
    if len(group_indices) > 0:
        quantile_errors.append(errors[group_indices].mean())


In [ ]:
# Plot quantile analysis
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(quantile_errors) + 1), quantile_errors, marker='o', linewidth=2)
plt.xlabel('Quantile', fontweight='bold', size=12)
plt.ylabel('Average Error', fontweight='bold', size=12)
plt.title('GCN Quantile Analysis', fontweight='bold', size=16)
plt.grid(True, alpha=0.3)
plt.savefig('gcn_quantile_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# SHAP ANALYSIS 

In [ ]:
print("Performing SHAP analysis...")

try:
    import shap
    
    # Sample a subset for SHAP analysis (computationally expensive)
    n_shap_samples = min(100, len(test_data))
    shap_indices = np.random.choice(len(test_data), n_shap_samples, replace=False)
    shap_data = [test_data[i] for i in shap_indices]
    
    # Create a wrapper function for SHAP that handles graph data
    def model_predict_wrapper(X_flat):
        """Wrapper function to convert flattened input back to graph format for SHAP."""
        predictions = []
        
        # Convert flattened features back to graph format
        for i in range(X_flat.shape[0]):
            # Reshape back to [n_features, window_size]
            node_features = X_flat[i].reshape(n_features, window_size)
            
            # Create a sample graph data object
            sample_graph = shap_data[0]  # Use first sample as template
            data = Data(
                x=torch.tensor(node_features, dtype=torch.float32),
                edge_index=sample_graph.edge_index,
                batch=torch.zeros(n_features, dtype=torch.long)  # Single graph
            ).to(device)
            
            # Get prediction
            model.eval()
            with torch.no_grad():
                pred = model(data)
                predictions.append(pred.cpu().numpy())
        
        return np.array(predictions)
    
    # Prepare data for SHAP
    # Flatten the graph node features for SHAP analysis
    background_data = []
    test_sample_data = []
    
    for i in range(min(50, len(train_data))):  # Background samples
        node_features = train_data[i].x.numpy()  # [n_features, window_size]
        background_data.append(node_features.flatten())
    
    for i in range(min(20, len(shap_data))):  # Test samples for explanation
        node_features = shap_data[i].x.numpy()
        test_sample_data.append(node_features.flatten())
    
    background_data = np.array(background_data)
    test_sample_data = np.array(test_sample_data)
    
    # Create SHAP explainer
    print("Creating SHAP explainer (this may take a while)...")
    explainer = shap.KernelExplainer(model_predict_wrapper, background_data[:10])  # Use fewer background samples
    
    # Calculate SHAP values
    print("Calculating SHAP values...")
    shap_values = explainer.shap_values(test_sample_data[:5])  # Analyze fewer test samples
    
    # Create feature names for visualization
    feature_names = []
    for i, col in enumerate(dataset.columns):
        for t in range(window_size):
            feature_names.append(f"{col}_t{t}")
    
    # Ensure feature names match the flattened data
    if len(feature_names) != test_sample_data.shape[1]:
        feature_names = [f"Feature_{i}" for i in range(test_sample_data.shape[1])]
    
    # Plot SHAP summary
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, test_sample_data[:5], 
                     feature_names=feature_names, show=False, max_display=20)
    plt.title('GCN SHAP Feature Importance', fontweight='bold', size=14)
    plt.tight_layout()
    plt.savefig('gcn_shap_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("SHAP analysis completed successfully!")
    
except ImportError:
    print("SHAP not available. Install with: pip install shap")
except Exception as e:
    print(f"SHAP analysis failed: {e}")
    print("This is normal for complex graph models - SHAP analysis is optional")

# SAVE RESULTS

In [ ]:
# Save predictions
results_df = pd.DataFrame({
    'actual': targets,
    'predicted': predictions,
    'error': errors
})
results_df.to_csv('gcn_predictions.csv', index=False)

In [ ]:
# Save metrics
metrics_dict = {
    'MSE': mse,
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'Training_Time': training_time,
    'Best_Val_Loss': best_val_loss
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv('gcn_metrics.csv', index=False)

In [ ]:
print(f"\n✅ GCN analysis completed successfully!")
print(f"📁 Results saved in current directory")
print(f"📊 Predictions: 'gcn_predictions.csv'")
print(f"📈 Visualizations: 'gcn_results.png', 'gcn_quantile_analysis.png'")
print(f"🏆 Model saved: 'best_gcn_model.pth'")
print(f"📋 Metrics: 'gcn_metrics.csv'")